# 2D → 3D Pipeline — Colab **Run all** (P6)

**Runtime ▸ Change runtime type ▸ T4 GPU** → **Runtime ▸ Run all**. Xong, không cần bấm gì thêm.

| Cell | Việc |
| :--: | :--- |
| 1 | Clone repo + cài dependencies + **DUSt3R thật (P2)** + TripoSR |
| 2 | Tải 8 ảnh của 1 vật từ **DX.GL Objaverse-1K** (196 góc/vật, phủ cả mặt cầu, có sẵn mask) |
| 3 | Bật FastAPI + Cloudflare Tunnel → in URL công khai |
| 4 | Chạy thử pipeline + **in TOÀN BỘ log P1→P5** |

Run all xong: mở URL ở Cell 3 → kéo 4–8 ảnh vào Web UI.

Ảnh test: **DX.GL Objaverse-1K** (196 góc/vật, phủ cả mặt cầu ±89°).
⚠️ **Nền ảnh là TRẮNG, không trong suốt** (README của bộ nói sai) — mask nằm ở thư mục `masks/`
RIÊNG, nên Cell 2 gắn mask đó vào kênh alpha trước khi gửi pipeline.

Vì sao Cell 2 lấy K ảnh chứ không lấy cả 196: DUSt3R chạy **N(N−1)/2 cặp** ảnh —
8 ảnh = 28 cặp, 16 = 120, 196 = **19.110 cặp** (~3–5 giờ GPU, dễ tràn RAM), mà chất lượng
đã bão hoà từ ~8 ảnh (2 camera gần nhất: K=4 → 94°, K=8 → 56°, K=12 → 45°, K=16 → 40°).

Bộ cũ `data/gso/` (5 góc một vòng ngang) vẫn dùng được nhưng **sẽ hở mặt đáy** vì không ai thấy mặt đó.

**Hết quota T4?** Runtime CPU vẫn chạy được — notebook tự phát hiện và giảm việc
(4 ảnh thay vì 8, alignment 100 vòng thay vì 300), chậm hơn nhiều nhưng ra cùng kết quả.
Muốn GPU nhanh mà không đợi quota: **Kaggle Notebooks** cho ~30 giờ T4/tuần, miễn phí.
Chạy trên Kaggle chỉ cần: bật *Internet* trong Settings, đổi `os.chdir('/content')` -> `os.chdir('/kaggle/working')`,
và bỏ `from google.colab import userdata` (đã nằm trong try/except nên tự bỏ qua).
Ảnh trong `data/input/multi_view/` là ảnh **VẼ bằng code** — không có dịch chuyển thật giữa các góc,
nên DUSt3R đoán sai camera và cho ra hình dẹt. Đừng dùng chúng để đánh giá chất lượng.

> **Web UI chạy qua tunnel**: pipeline 1–5 phút, nhưng Cloudflare cắt request sau ~100s.
> Nên Web UI **không** gọi thẳng `/generate-3d/` mà gọi `/generate-3d/job/` (trả `job_id` ngay)
> rồi hỏi trạng thái mỗi 2s. Nếu vẫn thấy `Unexpected token '<', "<!DOCTYPE "...`,
> nghĩa là có request dài lọt ra ngoài và đang bị proxy cắt — copy log gửi lại.
>
> ⚠️ Nếu Cell 1 báo `DUSt3R chưa import được` thì P2 rơi về MOCK — **hình dạng là NGẪU NHIÊN**.
> Đừng chụp kết quả, copy log Cell 4 gửi lại.


In [ ]:
# Cell 1: Clone repo (nhánh P6-tsdf-fix) + deps + DUSt3R thật (P2) + TripoSR
# Colab đã có torch GPU sẵn — KHÔNG cài lại (đè bản Colab -> vỡ CUDA runtime).
import os, sys, shutil

os.chdir('/content')   # kernel có thể đang đứng trong thư mục đã bị xoá -> mọi lệnh ! sau đó fail 'getcwd'

REPO, BRANCH = '/content/Img2d-to-3d', 'P6-tsdf-fix'   # PHẢI là nhánh chứa chính notebook này
if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)      # dọn thư mục rỗng còn sót từ lần chạy trước
    !git clone -q --branch {BRANCH} https://github.com/dduy26/Img2d-to-3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH}
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} pull -q
os.chdir(REPO)
!git log --oneline -1
!ls notebook/backend/app.py

!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx scikit-image opencv-python-headless kornia transformers xatlas

# ── DUSt3R thật (P2): hình dạng 3D lấy từ ảnh ──
if not os.path.isdir('/content/dust3r/.git'):
    shutil.rmtree('/content/dust3r', ignore_errors=True)
    !git clone -q --recursive https://github.com/naver/dust3r.git /content/dust3r
# KHÔNG chạy requirements.txt của dust3r: nó kéo torch/gradio/tensorboard, đè bản Colab -> vỡ CUDA
!pip install -q roma tqdm matplotlib einops safetensors

sys.path.append('/content/dust3r')
try:
    import dust3r  # noqa: F401
    print('✅ DUSt3R OK -> P2 chạy THẬT (hình dạng lấy từ ảnh)')
except Exception as e:
    print('⚠️ DUSt3R chưa import được -> P2 MOCK (hình dạng NGẪU NHIÊN):', e)

# ── TripoSR (chỉ cần cho chế độ 1 ảnh / nhánh Quality FAIL) ──
if not os.path.isdir('/content/TripoSR/.git'):
    shutil.rmtree('/content/TripoSR', ignore_errors=True)
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
# KHÔNG chạy requirements.txt của TripoSR: ghim transformers==4.35.0 (đè bản RMBG-2.0 cần)
!pip install -q omegaconf einops imageio
!pip install -q torchmcubes 2>/dev/null || echo 'torchmcubes thiếu (Colab là Python 3.13) -> TripoSR tự CPU fallback'
os.system(f'rm -rf {REPO}/notebook/backend/tsr && cp -r /content/TripoSR/tsr {REPO}/notebook/backend/tsr')

# ── RMBG-2.0 (tách nền) — repo GATED, cần token. CHỈ dùng khi ảnh KHÔNG có kênh alpha. ──
# Ảnh render có nền trong suốt (Cell 2 của bộ Objaverse-1K) thì P1 lấy luôn alpha làm mask
# -> chính xác hơn và KHÔNG cần token. Token chỉ cần cho ảnh chụp thật (.jpg nền trơn).
# Không có token thì P1 trả mask toàn 1 (không tách nền); pipeline vẫn chạy bình thường.
# Đã ĐO trên 5 ảnh GSO (nền trơn): mask chỉ đổi 4% thể tích mesh, vì P4 đã tự lọc nền
# bằng ngưỡng confidence của DUSt3R. Nền càng rối thì mask càng có ích.
# Bật (1 lần): 🔑 sidebar trái Colab -> Add new secret | Name: HF_TOKEN | Value: token của bạn
#              -> bật "Notebook access". Và bấm Agree ở https://huggingface.co/briaai/RMBG-2.0
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN đã nạp -> P1 tách nền thật bằng RMBG-2.0')
except Exception:
    print('ℹ️ Không có HF_TOKEN -> P1 trả mask toàn 1 (không tách nền). Vẫn chạy bình thường.')

print('Deps OK | cwd =', os.getcwd())


In [ ]:
# Cell 2: Ảnh test — DX.GL Objaverse-1K: 196 góc/vật, phủ CẢ MẶT CẦU (±89°).
# VÌ SAO ĐỔI BỘ DỮ LIỆU: bộ GSO cũ chỉ có 5 ảnh quanh MỘT VÒNG NGANG -> không camera nào thấy
# mặt đáy. Mặt không ai thấy thì không dựng được (TSDF là fusion thuần hình học) -> mesh hở đáy.
# Đo bằng pointmap chính xác tuyệt đối: 6 camera một vòng ngang 20° -> mesh chỉ ĐẶC 20% và mặt
# cắt ngang tách thành 5 đường bao (trông như "dán đôi"); 6 camera TẢN ĐỀU mặt cầu -> ĐẶC 99.8%,
# mặt cắt 1 đường bao. Khác nhau ở GÓC CHỤP, không ở SỐ ẢNH.
#
# SỰ THẬT ĐÃ ĐO VỀ BỘ NÀY (đừng tin README — README nói "nền trong suốt", SAI):
#   images/*.png    = RGB, NỀN TRẮNG (255,253,255). KHÔNG có kênh alpha.
#   masks/*.png     = mask nền, chứa ở KÊNH MÀU (R=G=B), kênh alpha của nó = 255 (vô dụng).
#   depth_16bit/*.png + transforms.json = độ sâu + pose CHÍNH CHỦ (dựng lại đúng hình vật,
#   tỉ lệ khớp points3D.ply -> dùng làm ground truth khi cần chẩn đoán P4).
#   => Cell này GẮN mask vào kênh alpha rồi lưu RGBA. P1 thấy alpha có sẵn -> dùng luôn làm
#      mask: chính xác từng pixel, KHÔNG cần RMBG-2.0, KHÔNG cần HF_TOKEN.
#      Không gắn thì ~88% nền TRẮNG bị coi là vật thể và bake thành màu texture.
import os, json, glob, shutil, zipfile, urllib.request
import numpy as np
from PIL import Image

os.chdir('/content/Img2d-to-3d')

# ── SỐ ẢNH: vặn ở đây ───────────────────────────────────────────────────────
#   K = 0    -> DÙNG HẾT mọi ảnh trong bộ (196). Tự chuyển sang graph THƯA "swin-3".
#   K = 4..16 -> chọn K ảnh TẢN MÁT NHẤT + graph "complete" (N(N-1) cặp).
#   K = 17..195 -> KHÔNG hợp lệ, xem chú thích ở khối kiểm tra bên dưới.
#
# VÌ SAO "DÙNG HẾT" KHÁC VỀ BẢN CHẤT: DUSt3R ghép TỪNG CẶP ảnh, nên chi phí là O(N²).
#   Với N=196 thì graph "complete" = 196*195 = 38.220 cặp -> ~250 GB RAM -> OOM ngay,
#   không phải "chậm". Nên khi dùng hết ảnh, Cell này chuyển sang graph THƯA "swin-3":
#   mỗi ảnh chỉ ghép với 3 ảnh liền trước + 3 liền sau trong danh sách -> 1.176 cặp (~8 GB).
#   Nhưng swin CHỈ đúng nếu ảnh liền kề trong danh sách là góc nhìn liền kề. Bộ dữ liệu gốc
#   xếp theo Fibonacci spiral: 2 khung LIỀN KỀ cách nhau tới 83° (đo được, trung vị 83°)
#   -> swin trên thứ tự gốc là VÔ NGHĨA. Nên Cell này SẮP LẠI theo đường đi "luôn nhảy tới
#   ảnh gần nhất"; sau khi sắp, láng giềng liền kề cách nhau ~15° và mỗi ảnh có đúng bậc 6.
K = 0
# SWIN: bậc đồ thị = 2*SWIN. Chỉ dùng khi K=0. Đây là nấc vặn RAM <-> chất lượng, ĐO trên
# chính bộ 196 ảnh này (thứ tự đường-đi-láng-giềng):
#     SWIN=2 ->   784 cặp | 6.4 GB | trung vị 17°/cặp | p95 34° | 4 cặp >90° | bậc 4  <- Colab free
#     SWIN=3 -> 1.176 cặp | 9.6 GB | trung vị 21°    | p95 51° | 9 cặp >90° | bậc 6
#     SWIN=4 -> 1.568 cặp | 12.9 GB | trung vị 29°   | p95 67° | 15 cặp >90°| bậc 8  <- cần 30GB RAM
# OOM giữa chừng -> giảm SWIN. Có RAM dư (Kaggle 30GB, Colab Pro high-RAM) -> tăng lên 3-4.
SWIN = 2
SLUG = '01'      # '01' = drone (РБВЗ-01) | shoe_3d_model | chair | toy_dog
# 1026 vật + tên zip: https://huggingface.co/datasets/dxgl/objaverse-1k (CC-BY 4.0)
LOCAL = 'data/01'         # bộ ĐÃ GIẢI NÉN (transforms.json + images/ + masks/): dùng luôn
OUT   = 'data/objaverse'  # Cell 4 đọc thư mục này TRƯỚC data/gso
PAIRS_PER_VIEW = 2 * SWIN  # bậc đồ thị khi dùng graph thưa (= 2 * winsize của swin)


def pick_spread(frames, k):
    """Chọn k camera TẢN MÁT NHẤT trên mặt cầu (farthest-point sampling).
    Không cần biết trục nào là 'trên': tự động lấy cả góc cao lẫn góc thấp."""
    centers = np.array([f['transform_matrix'] for f in frames], dtype=float)[:, :3, 3]
    dirs = centers - centers.mean(0)
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    taken = np.zeros(len(dirs), dtype=bool)
    chosen = [int(np.argmax(np.linalg.norm(dirs - dirs.mean(0), axis=1)))]
    taken[chosen] = True
    while len(chosen) < k:
        gap = (1.0 - dirs @ dirs[chosen].T).min(axis=1)   # 1 - cos = khoảng cách góc
        gap[taken] = -1.0                                 # đã chọn rồi thì bỏ
        nxt = int(np.argmax(gap))                         # xa tập đã chọn nhất
        chosen.append(nxt)
        taken[nxt] = True
    return sorted(chosen)
    # BẢN CŨ SAI: `gap[:, chosen] = -1.0` — dùng chỉ số KHUNG HÌNH làm chỉ số CỘT, mà gap chỉ
    # có len(chosen) cột -> IndexError ngay vòng lặp 2 (Cell 2 chết, chưa từng chạy được).


def order_by_proximity(frames):
    """Sắp ảnh thành đường đi 'luôn nhảy tới ảnh CHƯA thăm gần nhất' (greedy NN tour).

    Cần hàm này vì graph thưa `swin-k` của DUSt3R chỉ ghép các ảnh LIỀN KỀ TRONG DANH SÁCH.
    Bộ DX.GL xếp theo Fibonacci golden-angle spiral nên khung i và i+1 cách nhau tới 83°.
    Sau khi sắp: láng giềng liền kề ~15°, và swin-3 cho trung vị 21°/cặp, bậc đồ thị = 6.
    """
    C = np.array([f['transform_matrix'] for f in frames], dtype=float)[:, :3, 3]
    C = C - C.mean(0)
    D = C / np.linalg.norm(C, axis=1, keepdims=True)
    unvis = set(range(len(D)))
    cur = 0
    unvis.discard(cur)
    order = [cur]
    while unvis:
        u = list(unvis)
        cur = u[int(np.argmax(D[u] @ D[cur]))]     # gần ảnh hiện tại nhất
        order.append(cur)
        unvis.discard(cur)
    return order


try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
print(('🚀 GPU: có' if HAS_GPU else '🐢 KHÔNG có GPU -> CPU (rất chậm với bộ nhiều ảnh)'))


# ── NGUỒN ẢNH: ưu tiên bộ đã giải nén, không thì tải zip từ HuggingFace ──────
zip_handle = None
if os.path.exists(f'{LOCAL}/transforms.json'):
    frames = json.load(open(f'{LOCAL}/transforms.json'))['frames']
    print(f'✅ Dùng bộ ĐÃ GIẢI NÉN ở {LOCAL}/ ({len(frames)} góc) — không tải zip')
else:
    zp = f'/content/{SLUG}.zip'
    if not os.path.exists(zp):      # giữ zip: đổi K rồi chạy lại Cell này là xong ngay
        url = f'https://huggingface.co/datasets/dxgl/objaverse-1k/resolve/main/datasets/{SLUG}.zip'
        print(f'Đang tải {SLUG}.zip (~130 MB = 196 góc 1024x1024 + mask + depth + pose)...')
        urllib.request.urlretrieve(url, zp)
    zip_handle = zipfile.ZipFile(zp)
    frames = json.loads(zip_handle.read('transforms.json'))['frames']
    print(f'✅ Dùng zip {zp} ({len(frames)} góc)')


def _member(kind, name):
    """Mở 1 file trong bộ dữ liệu, dù nguồn là zip hay thư mục đã giải nén."""
    return zip_handle.open(f'{kind}/{name}') if zip_handle else open(f'{LOCAL}/{kind}/{name}', 'rb')


STILL = len(np.unique(np.round(
    np.array([f['transform_matrix'] for f in frames], float)[:, :3, 3], 4), axis=0))
if STILL < len(frames):
    print(f'ℹ️ {STILL}/{len(frames)} tư thế camera phân biệt — bộ này có khung trùng.')

# ── CHỌN SỐ ẢNH + GRAPH ──────────────────────────────────────────────────────
n_all = min(STILL, len(frames))
if K == 0:
    idx, scene_graph = order_by_proximity(frames), f'swin-{SWIN}'
elif K <= 16:
    idx, scene_graph = pick_spread(frames, K), 'complete'
else:
    # "complete" với N lớn là OOM chứ không phải chậm — chặn trước, đừng để nó chết giữa chừng.
    raise ValueError(
        f'K={K} với graph "complete" cần {K*(K-1):,} cặp (~{K*(K-1)*8.6/1024:.0f} GB RAM) -> OOM.\n'
        f'  Chọn K = 0 để DÙNG HẾT {n_all} ảnh (graph thưa "swin-3", ~8 GB), hoặc K <= 16.'
    )
K = len(idx)

shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)
for n, i in enumerate(idx, 1):
    name = os.path.basename(frames[i]['file_path'])              # frame_00042.png
    rgb = Image.open(_member('images', name)).convert('RGB')
    mask = Image.open(_member('masks', name)).convert('L')       # mask nằm ở KÊNH MÀU
    rgb.putalpha(mask)                                           # -> RGBA: P1 dùng alpha làm mask
    rgb.save(f'{OUT}/view_{n:02d}.png')
if zip_handle:
    zip_handle.close()

# ── KIỂM CHỨNG: danh sách láng giềng có thật sự liền kề theo GÓC không? ──────
D = np.array([frames[i]['transform_matrix'] for i in idx], float)[:, :3, 3]
D = D - D.mean(0)
D /= np.linalg.norm(D, axis=1, keepdims=True)
ang = np.degrees(np.arccos(np.clip(D @ D.T, -1, 1)))
np.fill_diagonal(ang, 180.0)
nb = np.min(np.stack([ang[np.arange(K), (np.arange(K) + d) % K] for d in (1, -1)]), axis=0)
print(f'→ {K}/{len(frames)} ảnh -> {OUT}/  (vật "{SLUG}") | graph = {scene_graph}')
if K > 2:
    print(f'→ 2 camera GẦN NHAU NHẤT trong cả bộ: {ang.min():.0f}°')
    print(f'→ láng giềng LIỀN KỀ: trung vị {np.median(nb):.0f}°, xấu nhất {nb.max():.0f}°'
          f'  (<30° là tốt; gần 180° = swin sẽ ghép 2 ảnh chẳng liên quan)')
if idx:
    C = np.array([frames[i]['transform_matrix'] for i in idx], float)[:, :3, 3]
    if len(C) > 1:
        d0 = C - C.mean(0); d0 /= np.linalg.norm(d0, axis=1, keepdims=True)
        a2 = np.degrees(np.arccos(np.clip(d0 @ d0.T, -1, 1))); a2[np.arange(len(C)), np.arange(len(C))] = 180
        print(f'→ 2 camera XA NHAU NHẤT: {a2.max():.0f}°  (>=120° là phủ tốt cả mặt cầu)')
n_cap = K * (K - 1) if scene_graph == 'complete' else PAIRS_PER_VIEW * K
print(f'→ {n_cap:,} cặp ảnh cho DUSt3R  |  ~{n_cap*8.6/1024:.0f} GB RAM đỉnh')
print(f'→ mỗi ảnh ĐÃ GẮN mask nền vào kênh alpha -> P1 dùng luôn, không cần HF_TOKEN/RMBG')
open('/content/views.txt', 'w').write(str(K))          # Cell 3 & 4 đọc
open('/content/scene.txt', 'w').write(scene_graph)     # Cell 3 đọc -> env DUST3R_SCENE_GRAPH
print('cwd =', os.getcwd())


In [ ]:
# Cell 3: Bật FastAPI + Cloudflare Tunnel (KHÔNG cần tài khoản Cloudflare)
import subprocess, time, os, re, socket, urllib.request

REPO    = '/content/Img2d-to-3d'                            # tuyệt đối, KHÔNG phụ thuộc cwd
BACKEND = os.path.join(REPO, 'notebook', 'backend')         # app.py tạo temp_uploads/ & outputs/ theo cwd
assert os.path.isdir(BACKEND), f'Không thấy {BACKEND} -> chạy lại Cell 1'
os.chdir(REPO)
PORT = 8000

# ── Dọn server/tunnel của lần chạy TRƯỚC ──
# KHÔNG dùng `pkill -f` qua shell: chính dòng lệnh đó chứa chuỗi cần tìm nên nó khớp luôn
# với shell đang chạy nó -> tự giết shell -> cell quay MÃI. (Và nếu image thiếu psmisc thì
# `pkill` không tồn tại -> im lặng không giết gì -> port 8000 vẫn bị chiếm -> Errno 98.)
# Đọc /proc trực tiếp: không cần binary ngoài, và bỏ qua chính nó + tổ tiên nên không tự giết.
def kill_stale(*keywords):
    me = {os.getpid(), os.getppid()}
    victims = []
    if not os.path.isdir('/proc'):      # Colab luôn có; guard để không vỡ ở nơi khác
        return victims
    for name in os.listdir('/proc'):
        if not name.isdigit() or int(name) in me:
            continue
        try:
            cmd = open(f'/proc/{name}/cmdline', 'rb').read().replace(b'\0', b' ').decode('utf-8', 'replace')
        except Exception:
            continue
        if not cmd or 'ipykernel' in cmd or 'jupyter' in cmd:
            continue
        if any(k in cmd for k in keywords):
            try:
                os.kill(int(name), 15)          # SIGTERM trước: uvicorn tự tắt gọn
                victims.append((int(name), cmd[:70]))
            except Exception as e:
                print(f'  không tắt được pid {name}: {e}')
    return victims

def kill_hard(victims, wait=3):
    """SIGTERM không ăn thì SIGKILL — server cũ ôm port 8000 là nguồn của Errno 98."""
    if not victims:
        return
    time.sleep(wait)
    for pid, cmd in victims:
        try:
            os.kill(pid, 0)                     # còn sống?
            os.kill(pid, 9)                     # SIGKILL
            print(f'  SIGKILL pid {pid}: {cmd}')
        except ProcessLookupError:
            print(f'  đã tắt pid {pid}: {cmd}')
        except Exception as e:
            print(f'  không tắt được pid {pid}: {e}')

def port_free(p):
    with socket.socket() as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind(('0.0.0.0', p)); return True
        except OSError:
            return False

print('Dọn tiến trình cũ (nếu có):')
kill_hard(kill_stale('uvicorn', 'cloudflared'))   # tat han, ke ca khi SIGTERM bi bo qua
time.sleep(2)                                   # nhường port
if not port_free(PORT):                         # hiếm: chưa kịp nhả -> đợi thêm, rồi đổi port
    time.sleep(4)
if not port_free(PORT):
    print(f'⚠️ Port {PORT} vẫn bị chiếm -> chuyển sang {PORT+1}')
    PORT += 1
print(f'Port dùng: {PORT}')
open('/content/port.txt', 'w').write(str(PORT))    # Cell 4 đọc lại, khỏi hardcode

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

env = dict(os.environ)
env['PYTHONPATH'] = '/content/dust3r' + os.pathsep + env.get('PYTHONPATH', '')
# Nút vặn chất lượng.
#   DUST3R_NITER: 100 -> 0.16067 | 300 -> 0.17599 (Chamfer, càng nhỏ càng tốt).
#     Ít vòng lại chính xác hơn, nhưng 300 an toàn hơn -> để 300.
#   TSDF_RES: ĐO LẠI BẰNG GROUND TRUTH (196 góc thật của DX.GL, depth+pose chính chủ):
#     cho P4 ăn pointmap ĐÚNG, rồi so bề rộng mesh với chính điểm đưa vào:
#        128 -> tỉ lệ trục [1.00, 0.58, 0.50]  (điểm vào [1.00, 0.89, 0.73]) -> HAO 28%
#        192 -> [1.00, 0.87, 0.82]   <- hết hao
#        256 -> [1.00, 0.87, 0.80]   (không hơn 192, chỉ tốn RAM)
#     Làm mượt Taubin KHÔNG gây ra hao này (số đỉnh y hệt ở mọi mức làm mượt).
#     Vật càng mỏng/thưa (drone, cánh, chân ghế) càng cần res cao. Đặt 192.
env.setdefault('TSDF_RES', '192')
# PREPROC_MAX_IMAGES = K mà Cell 2 đã chọn. Trần mặc định của preprocess.py là 8, nạp quá trần
# thì bị CẮT XUỐNG 6 theo CHỈ SỐ -> K ảnh tản mát vừa chọn bị thay bằng 6 ảnh chọn bừa.
try:
    _k = int(open('/content/views.txt').read().strip())
except Exception:
    _k = 8
env['PREPROC_MAX_IMAGES'] = str(_k)
env.setdefault('PREPROC_TARGET_COUNT', str(_k))
print(f'-> preprocess.py sẽ nhận tối đa {_k} ảnh (PREPROC_MAX_IMAGES)')
# GRAPH CẶP ẢNH của DUSt3R = chi phí thật của P2. Cell 2 chọn: K nhỏ -> "complete",
# dùng hết ảnh -> "swin-3" (O(N) cặp). KHÔNG tự đổi ở đây; chỉ chuyển tiếp lựa chọn đó.
try:
    _scene = open('/content/scene.txt').read().strip()
except Exception:
    _scene = 'complete'
env['DUST3R_SCENE_GRAPH'] = _scene
# batch_size=1 cứng nghĩa là mỗi cặp một lượt forward. 1.176 cặp mà batch 1 thì cực chậm;
# T4 15GB chứa được 4. Graph "complete" ít cặp hơn nên để 1 cho chắc.
env.setdefault('DUST3R_BATCH', '1' if _scene == 'complete' else '4')
print(f'-> DUSt3R: scene_graph={_scene}, batch_size={env["DUST3R_BATCH"]}')
# DUST3R_NITER: trên CPU 300 vòng global alignment mất hàng chục phút.
#   Đo trước đây trên bộ ảnh này: 100 -> Chamfer 0.16067 | 300 -> 0.17599 (nhỏ hơn = tốt hơn),
#   tức 100 KHÔNG phải là thoả hiệp chất lượng, chỉ là ít vòng hơn. CPU dùng 100.
try:
    import torch as _torch
    _has_gpu = _torch.cuda.is_available()
except Exception:
    _has_gpu = False
print(('🚀 GPU: có' if _has_gpu else '🐢 KHÔNG có GPU -> CPU (chậm hơn nhiều, kiên nhẫn)'))
env.setdefault('DUST3R_NITER', '300' if _has_gpu else '100')
# TSDF_TRUNC_FRAC: mu cắt ngắn = tỉ lệ này × cạnh lớn nhất của vật. KHÔNG theo resolution.
#   (Đo Chamfer: buộc mu vào voxel_size làm res cao LỆCH hơn, nên đã tách ra.)
#   To hơn -> bề mặt mượt hơn nhưng mất chi tiết mỏng. Thử 0.01 / 0.02 / 0.04.
# TSDF_SMOOTH_ITER: số vòng làm mượt Taubin sau Marching Cubes. 0 = tắt. Thử 0 / 5 / 15.
env.setdefault('TSDF_TRUNC_FRAC', '0.02')
env.setdefault('TSDF_SMOOTH_ITER', '5')

# Log ra FILE, KHÔNG dùng PIPE: pipe không ai đọc -> đầy 64KB -> tiến trình con TẮC giữa chừng.
server = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', str(PORT)],
    cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT, text=True,
)

def read_log(n=4000):
    """Log cuối của server, để biết vì sao chết."""
    try:
        return open('/content/server.log', encoding='utf-8', errors='replace').read()[-n:]
    except Exception as e:
        return f'(chưa có log: {e})'

# Chờ server READY. Lần đầu nạp DUSt3R ~2.3GB nên có thể vài phút -> in tiến độ,
# VÀ thoát NGAY nếu tiến trình đã chết (đừng ngồi chờ đủ 5 phút trong im lặng).
ready = False
for i in range(150):
    if server.poll() is not None:
        print(f'❌ Server THOÁT ngay (mã {server.returncode}) sau ~{i*2}s. Log cuối:')
        print(read_log())
        break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=2) as r:
            print('Server READY:', r.read().decode()); ready = True; break
    except Exception:
        pass
    if i % 5 == 4:
        print(f'  ...đang nạp model ({(i+1)*2}s) — xem tiếp: !tail -5 /content/server.log', flush=True)
    time.sleep(2)
else:
    print('⏱ Hết 5 phút mà server chưa READY. Log cuối:')
    print(read_log())

if ready:
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
        stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT, text=True,
    )
    url = None
    for _ in range(60):             # đọc FILE, không đọc pipe -> cloudflared không bao giờ tắc
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com',
                      open('/content/tunnel.log', encoding='utf-8', errors='replace').read())
        if m:
            url = m.group(0); print('\n🌐 WEB UI  :', url); print('📘 SWAGGER :', url + '/docs'); break
        time.sleep(1)
    if url is None:
        print('Không lấy được URL tunnel — xem /content/tunnel.log')


In [ ]:
# Cell 4: Chạy thử pipeline + in TOÀN BỘ log P1→P5
# Gọi thẳng 127.0.0.1 (không qua tunnel) nên chạy bao lâu cũng được — dùng được endpoint sync.
# Nếu có lỗi: copy nguyên phần "LOG SERVER" bên dưới gửi lại.
import glob, subprocess, json, os

os.chdir('/content/Img2d-to-3d')

PORT = int(open('/content/port.txt').read().strip()) if os.path.exists('/content/port.txt') else 8000
BASE = f'http://127.0.0.1:{PORT}'

obj = sorted(glob.glob('/content/Img2d-to-3d/data/objaverse/view_*.png'))         # PHỦ MẶT CẦU (Cell 2)
gso = sorted(glob.glob('/content/Img2d-to-3d/data/gso/view_*.jpg'))               # chỉ 1 vòng ngang
mv  = sorted(glob.glob('/content/Img2d-to-3d/data/input/multi_view/view_*.jpg'))  # VẼ (không parallax)
imgs = obj if len(obj) >= 4 else (gso if len(gso) >= 4 else mv)
assert imgs, 'Không có ảnh test -> chạy lại Cell 2'
if not obj:
    print('⚠️ Không có data/objaverse -> đang dùng', imgs[0])
    if not gso:
        print('   Ảnh vẽ KHÔNG có dịch chuyển thật giữa các góc -> DUSt3R đoán sai camera -> hình DẸT.')
print(f'Dùng {len(imgs)} ảnh:', [os.path.basename(p) for p in imgs[:8]])
print('⏳ Chờ tới khi in bảng KẾT QUẢ API. Ước lượng: GPU ~2–4 phút | CPU ~10–20 phút.')
print('   CPU: DUSt3R chạy từng CẶP ảnh + global alignment -> chậm, nhưng không bị cắt')
print('   vì cell này gọi thẳng 127.0.0.1 (không qua tunnel).')

# Số ảnh gửi đi = K mà Cell 2 chọn (Cell 3 đã nâng PREPROC_MAX_IMAGES = K cho vừa).
# Đừng gửi nhiều hơn: preprocess.py cắt phần dư xuống TARGET theo CHỈ SỐ, không theo góc.
_K = int(open('/content/views.txt').read().strip()) if os.path.exists('/content/views.txt') else 8
print(f'Gửi {min(len(imgs), _K)}/{len(imgs)} ảnh (K={_K} đặt ở Cell 2)')

if len(imgs) >= 2:
    cmd = ['curl', '-s', '-X', 'POST', BASE + '/generate-3d/']
    for p in imgs[:_K]:
        cmd += ['-F', f'files=@{p}']
else:
    cmd = ['curl', '-s', '-X', 'POST', BASE + '/generate-3d/single/', '-F', f'file=@{imgs[0]}']

r = subprocess.run(cmd, capture_output=True, text=True)
print('\n--- KẾT QUẢ API ---')
try:
    print(json.dumps(json.loads(r.stdout), indent=2, ensure_ascii=False))
except Exception:
    print('stdout:', r.stdout[:1000], '\nstderr:', r.stderr[-500:])

print('\n--- FILE .glb ---')
for f in sorted(glob.glob('/content/Img2d-to-3d/notebook/backend/outputs/*.glb')):
    print(f'{os.path.basename(f)}  {os.path.getsize(f)} bytes')

print('\n--- LOG SERVER (P1→P5) — copy từ đây nếu cần báo lỗi ---')
print(open('/content/server.log', encoding='utf-8', errors='replace').read()[-8000:])


In [ ]:
# (BỎ COMMENT rồi chạy cell này) — xem 40 vật đầu để chọn OBJ_INDEX cho Cell 2
# from datasets import load_dataset
# ds = load_dataset("suvadityamuk/google-scanned-objects", split="train", streaming=True)
# for i, s in enumerate(ds):
#     if i >= 40: break
#     m = s["json"]
#     print(i, (m.get('category') or m.get('category_name')), '|', m.get('name'))


## Dừng server

**Runtime ▸ Restart session** (đừng dùng `pkill -f uvicorn`: dòng lệnh đó tự khớp chính nó và treo cell). File `.glb` nằm ở
`notebook/backend/outputs/` — tải về bằng panel **Files** bên trái.

## Nguồn ảnh khác

| Nguồn | Đặc điểm |
| :--- | :--- |
| **DX.GL Objaverse-1K** (Cell 2, mặc định) | 1026 vật, **196 ảnh 1024×1024 mỗi vật**, phủ cả mặt cầu, kèm mask + poses + depth, CC-BY 4.0 |
| Google Scanned Objects | 1030 vật, chỉ **5 ảnh render quanh 1 vòng ngang** + `gt.glb` để so |
| OmniObject3D | 6000 vật / 190 nhóm, **100 ảnh 800×800** mỗi vật (cần đăng ký OpenDataLab) |
| CO3D (Meta) | 19k vật, ảnh chụp thật ngoài đời, có mask tách nền sẵn (bản nhỏ 8.9 GB) |
| DTU | Benchmark MVS, 49–64 góc, có ground-truth point cloud |

**Tốt nhất vẫn là ảnh tự chụp**: đặt vật lên nền trơn → đi vòng quanh, **8 tấm cách đều ~45°**,
giữ nguyên khoảng cách và độ cao máy, **không xoay vật**.
